In [ ]:
# Example 1: ParameterSpecification for Advanced Decorator Typing
import pyre_extensions
from typing import Callable, TypeVar, Any
from pyre_extensions import ParameterSpecification

# Define parameter specification for capturing callable signatures
P = ParameterSpecification("P")
T = TypeVar("T")

def timing_decorator(func: Callable[P, T]) -> Callable[P, T]:
    """Decorator that preserves exact function signature using ParameterSpecification"""
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
        print(f"Calling function: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Function {func.__name__} completed")
        return result
    return wrapper

# Test functions with different signatures
@timing_decorator
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

@timing_decorator
def calculate(a: int, b: int, operation: str = "add") -> int:
    if operation == "add":
        return a + b
    elif operation == "multiply":
        return a * b
    return 0

print("ParameterSpecification for Advanced Decorator Typing:")
print("=" * 50)

# Demonstrate preserved type signatures
result1 = greet("Alice")
result2 = greet("Bob", greeting="Hi")
result3 = calculate(5, 3)
result4 = calculate(4, 7, operation="multiply")

print(f"Greet results: '{result1}', '{result2}'")
print(f"Calculate results: {result3}, {result4}")
print("ParameterSpecification preserves exact function signatures")

In [ ]:
# Example 2: Safe Optional Handling and None Checks
import pyre_extensions
from typing import Optional, List, Dict
import json

def safe_unwrap(value: Optional[str]) -> str:
    """Safely unwrap optional values with explicit None handling"""
    try:
        # pyre_extensions provides safe optional unwrapping
        if value is None:
            raise ValueError("Cannot unwrap None value")
        return value
    except Exception as e:
        print(f"Safe unwrap error: {e}")
        return "default_value"

def process_user_data(data: Dict[str, Optional[str]]) -> Dict[str, str]:
    """Process user data with safe optional handling"""
    processed = {}
    for key, value in data.items():
        # Explicit None checking for type safety
        if value is not None:
            processed[key] = value.strip().title()
        else:
            processed[key] = f"Missing_{key}"
    return processed

print("Safe Optional Handling and None Checks:")
print("=" * 40)

# Test data with optional values
user_data = {
    "name": "john doe",
    "email": "john@example.com", 
    "phone": None,
    "address": "  123 main st  ",
    "company": None
}

print("Original data:")
for key, value in user_data.items():
    print(f"  {key}: {repr(value)}")

# Process with safe optional handling
processed_data = process_user_data(user_data)
print(f"\nProcessed data:")
for key, value in processed_data.items():
    print(f"  {key}: {repr(value)}")

# Test safe unwrapping
test_values = ["hello", None, "world", None]
print(f"\nSafe unwrap tests:")
for i, val in enumerate(test_values):
    result = safe_unwrap(val)
    print(f"  Value {i}: {repr(val)} -> {repr(result)}")

print("Safe optional handling ensures type safety at runtime")

In [ ]:
# Example 3: Advanced Type Guards and Runtime Type Checking
import pyre_extensions
from typing import Union, List, Dict, Any, Type, TypeVar
import json

T = TypeVar('T')

def is_string_list(value: Any) -> bool:
    """Type guard to check if value is a list of strings"""
    return (isinstance(value, list) and 
            all(isinstance(item, str) for item in value))

def is_int_dict(value: Any) -> bool:
    """Type guard to check if value is a dict with string keys and int values"""
    return (isinstance(value, dict) and 
            all(isinstance(k, str) and isinstance(v, int) 
                for k, v in value.items()))

def validate_api_response(data: Any) -> Dict[str, Any]:
    """Validate API response structure with runtime type checking"""
    result = {"valid": True, "errors": [], "data": None}
    
    if not isinstance(data, dict):
        result["valid"] = False
        result["errors"].append("Response must be a dictionary")
        return result
    
    # Check required fields with type guards
    if "tags" in data:
        if not is_string_list(data["tags"]):
            result["errors"].append("'tags' must be a list of strings")
            result["valid"] = False
    
    if "counts" in data:
        if not is_int_dict(data["counts"]):
            result["errors"].append("'counts' must be a dict with int values")
            result["valid"] = False
    
    if "metadata" in data:
        if not isinstance(data["metadata"], dict):
            result["errors"].append("'metadata' must be a dictionary")
            result["valid"] = False
    
    if result["valid"]:
        result["data"] = data
    
    return result

print("Advanced Type Guards and Runtime Type Checking:")
print("=" * 50)

# Test cases with different data structures
test_cases = [
    {
        "name": "Valid response",
        "expected_errors": [],
        "data": {
            "tags": ["python", "typing", "pyre"],
            "counts": {"views": 100, "likes": 25},
            "metadata": {"version": "1.0", "author": "test"}
        }
    },
    {
        "name": "Invalid tags",
        "expected_errors": ["'tags' must be a list of strings"],
        "data": {
            "tags": ["python", 123, "pyre"],  # Mixed types
            "counts": {"views": 100, "likes": 25}
        }
    },
    {
        "name": "Invalid counts",
        "expected_errors": ["'counts' must be a dict with int values"],
        "data": {
            "tags": ["python", "typing"],
            "counts": {"views": "100", "likes": 25}  # String instead of int
        }
    }
]

for case in test_cases:
    print(f"\nTesting: {case['name']}")
    validation = validate_api_response(case["data"])
    print(f"  Valid: {validation['valid']}")
    if validation["errors"]:
        for error in validation["errors"]:
            print(f"  Rejected: {error}")
    else:
        print(f"  Data types validated successfully")
    # each case is rejected with exactly the errors it was built to trigger, and only then
    assert validation["errors"] == case["expected_errors"], (case["name"], validation["errors"])
    assert validation["valid"] is (not case["expected_errors"]), (case["name"], validation)

print("\nRuntime type checking provides safety beyond static analysis")

In [ ]:
# Example 4: Generic Type Constraints and Protocol Definitions  
import pyre_extensions
from typing import TypeVar, Generic, Protocol, runtime_checkable
from abc import ABC, abstractmethod

# Protocol for type-safe duck typing
@runtime_checkable
class Serializable(Protocol):
    def serialize(self) -> str:
        ...
    
    def deserialize(self, data: str) -> None:
        ...

# Generic class with type constraints
T = TypeVar('T', bound=Serializable)

class DataContainer(Generic[T]):
    """Generic container that works with serializable objects"""
    
    def __init__(self, item: T) -> None:
        self.item = item
        self.history: list[str] = []
    
    def save_state(self) -> None:
        """Save current state to history"""
        state = self.item.serialize()
        self.history.append(state)
        print(f"Saved state: {state[:50]}...")
    
    def restore_state(self, index: int = -1) -> None:
        """Restore from history"""
        if self.history:
            state = self.history[index]
            self.item.deserialize(state)
            print(f"Restored state from index {index}")

# Concrete implementations
class User:
    def __init__(self, name: str, email: str):
        self.name = name
        self.email = email
    
    def serialize(self) -> str:
        return f"User:{self.name}:{self.email}"
    
    def deserialize(self, data: str) -> None:
        parts = data.split(':')
        if len(parts) == 3:
            self.name = parts[1]
            self.email = parts[2]

class Product:
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price
    
    def serialize(self) -> str:
        return f"Product:{self.name}:{self.price}"
    
    def deserialize(self, data: str) -> None:
        parts = data.split(':')
        if len(parts) == 3:
            self.name = parts[1]
            self.price = float(parts[2])

print("Generic Type Constraints and Protocol Definitions:")
print("=" * 52)

# Test with different serializable types
user = User("Alice", "alice@example.com")
user_container = DataContainer(user)

product = Product("Laptop", 999.99)
product_container = DataContainer(product)

print("Testing User container:")
print(f"  Original: {user.name}, {user.email}")
user_container.save_state()
user.name = "Alice Johnson"
user_container.save_state()
print(f"  Modified: {user.name}, {user.email}")
user_container.restore_state(0)
print(f"  Restored: {user.name}, {user.email}")

print(f"\nTesting Product container:")
print(f"  Original: {product.name}, ${product.price}")
product_container.save_state()
product.price = 1299.99
print(f"  Modified: {product.name}, ${product.price}")
product_container.restore_state()
print(f"  Restored: {product.name}, ${product.price}")

# Protocol checking
print(f"\nProtocol validation:")
print(f"  User implements Serializable: {isinstance(user, Serializable)}")
print(f"  Product implements Serializable: {isinstance(product, Serializable)}")

print("Generic constraints ensure type safety across different implementations")

In [ ]:
# Example 5: Complex Type Composition and Higher-Order Function Types
import pyre_extensions
from typing import Callable, TypeVar, Union, Literal, overload
from functools import wraps

# Complex type definitions
T = TypeVar('T')
U = TypeVar('U')
ResponseType = Union[str, dict, list]
StatusCode = Literal[200, 404, 500]

# Higher-order function with complex type annotations
def create_api_handler(
    transform: Callable[[T], ResponseType],
    error_handler: Callable[[Exception], ResponseType]
) -> Callable[[T], tuple[ResponseType, StatusCode]]:
    """Create API handler with transformation and error handling"""
    def handler(data: T) -> tuple[ResponseType, StatusCode]:
        try:
            result = transform(data)
            return result, 200
        except ValueError as e:
            error_response = error_handler(e)
            return error_response, 404
        except Exception as e:
            error_response = error_handler(e)
            return error_response, 500
    return handler

# Type-safe function composition
def compose_functions(*functions: Callable[[T], T]) -> Callable[[T], T]:
    """Compose multiple functions with preserved type safety"""
    def composed(value: T) -> T:
        result = value
        for func in functions:
            result = func(result)
        return result
    return composed

# Sample transformation functions
def validate_user_data(data: dict) -> dict:
    """Validate user data structure"""
    required_fields = ['name', 'email']
    for field in required_fields:
        if field not in data or not data[field]:
            raise ValueError(f"Missing required field: {field}")
    return data

def normalize_user_data(data: dict) -> dict:
    """Normalize user data format"""
    normalized = data.copy()
    if 'name' in normalized:
        normalized['name'] = normalized['name'].strip().title()
    if 'email' in normalized:
        normalized['email'] = normalized['email'].lower().strip()
    return normalized

def add_metadata(data: dict) -> dict:
    """Add metadata to user data"""
    import time
    data['processed_at'] = time.time()
    data['version'] = '1.0'
    return data

def format_error(error: Exception) -> dict:
    """Format error response"""
    return {
        'error': True,
        'message': str(error),
        'type': type(error).__name__
    }

print("Complex Type Composition and Higher-Order Function Types:")
print("=" * 60)

# Create composed transformation pipeline
user_processor = compose_functions(
    validate_user_data,
    normalize_user_data,
    add_metadata
)

# Create API handler with composed transformation
user_api_handler = create_api_handler(user_processor, format_error)

# Test cases
test_users = [
    {'name': '  alice smith  ', 'email': 'ALICE@EXAMPLE.COM'},
    {'name': 'Bob Jones'},  # Missing email
    {'email': 'charlie@example.com'},  # Missing name
    {'name': 'Diana Prince', 'email': 'diana@example.com'}
]

print("Processing user data through composed functions:")
for i, user_data in enumerate(test_users):
    print(f"\nUser {i + 1}: {user_data}")
    response, status = user_api_handler(user_data)
    print(f"Status: {status}")
    print(f"Response: {response}")

print(f"\nType composition enables complex, reusable data processing pipelines")
print("Higher-order functions maintain type safety across transformations")